In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_PATH:", SRC_PATH)

PROJECT_ROOT: /Users/amirhosseinlatifinavid/Desktop/Programming/Python /quant-ml-backtester
SRC_PATH: /Users/amirhosseinlatifinavid/Desktop/Programming/Python /quant-ml-backtester/src


In [2]:
import pandas as pd

from quant_ml.config import RAW_DATA_DIR
from quant_ml.data_checks import (
    summarize_dataframe,
    check_index_properties,
    missing_by_column,
    missing_by_row,
    flatten_yahoo_columns,
    duplicate_dates,
    date_gaps,
)

In [3]:
prices = pd.read_parquet(RAW_DATA_DIR / "yahoo_prices.parquet")
macro = pd.read_parquet(RAW_DATA_DIR / "fred_macro.parquet")

print("prices shape:", prices.shape)
print("macro shape:", macro.shape)

prices shape: (3519, 30)
macro shape: (950, 5)


In [4]:
prices.head()

Price           Close                                                          \
Ticker            EEM         GLD        IWM        QQQ        SPY        TLT   
Date                                                                            
2012-01-03  28.759422  155.919998  61.813828  50.312561  99.312202  81.165367   
2012-01-04  28.597609  156.710007  61.475780  50.524776  99.468002  80.200340   
2012-01-05  28.472559  157.779999  61.822060  50.940365  99.732788  80.057587   
2012-01-06  28.112152  157.199997  61.665424  51.117214  99.475777  80.689659   
2012-01-09  28.406364  156.500000  61.978748  50.949203  99.717232  80.546921   

Price            High                                    ...       Open  \
Ticker            EEM         GLD        IWM        QQQ  ...        IWM   
Date                                                     ...              
2012-01-03  28.884465  156.300003  62.555892  50.568985  ...  62.399232   
2012-01-04  28.663807  157.380005  61.723137  50.595516  ...  61.492274   
2012-01-05  28.560825  158.029999  62.118879  50.984576  ...  61.038768   
2012-01-06  28.443141  158.630005  62.151885  51.232161  ...  61.855059   
2012-01-09  28.465209  157.589996  62.085939  51.267526  ...  61.954014   

Price                                          Volume                      \
Ticker            QQQ        SPY        TLT       EEM       GLD       IWM   
Date                                                                        
2012-01-03  50.321402  99.514723  81.158570  72557500  13385800  60504700   
2012-01-04  50.232983  99.078543  81.049848  46225600  11549700  34648500   
2012-01-05  50.418670  98.930508  80.309038  54383000  11621600  57274600   
2012-01-06  50.949209  99.857445  80.023643  50333100   9790500  45499800   
2012-01-09  51.249841  99.701650  80.512944  45814800   8771900  52042400   

Price                                     
Ticker           QQQ        SPY      TLT  
Date                                      
2012-01-03  39514100  193697900  9076900  
2012-01-04  29403400  127186500  8417100  
2012-01-05  41260600  173895000  6465800  
2012-01-06  46325200  148050000  7348500  
2012-01-09  39195500   99530200  5582400  

[5 rows x 30 columns]

In [5]:
type(prices.columns), prices.columns[:10]

(pandas.core.indexes.multi.MultiIndex,
 MultiIndex([('Close', 'EEM'),
             ('Close', 'GLD'),
             ('Close', 'IWM'),
             ('Close', 'QQQ'),
             ('Close', 'SPY'),
             ('Close', 'TLT'),
             ( 'High', 'EEM'),
             ( 'High', 'GLD'),
             ( 'High', 'IWM'),
             ( 'High', 'QQQ')],
            names=['Price', 'Ticker']))

In [6]:
prices_flat = flatten_yahoo_columns(prices)
prices_flat.head()

,Close_EEM,Close_GLD,Close_IWM,Close_QQQ,Close_SPY,Close_TLT,High_EEM,High_GLD,High_IWM,High_QQQ,...,Open_IWM,Open_QQQ,Open_SPY,Open_TLT,Volume_EEM,Volume_GLD,Volume_IWM,Volume_QQQ,Volume_SPY,Volume_TLT
Date,,,,,,,,,,,,,,,,,,,,,
2012-01-03,28.759422,155.919998,61.813828,50.312561,99.312202,81.165367,28.884465,156.300003,62.555892,50.568985,...,62.399232,50.321402,99.514723,81.158570,72557500,13385800,60504700,39514100,193697900,9076900
2012-01-04,28.597609,156.710007,61.475780,50.524776,99.468002,80.200340,28.663807,157.380005,61.723137,50.595516,...,61.492274,50.232983,99.078543,81.049848,46225600,11549700,34648500,29403400,127186500,8417100
2012-01-05,28.472559,157.779999,61.822060,50.940365,99.732788,80.057587,28.560825,158.029999,62.118879,50.984576,...,61.038768,50.418670,98.930508,80.309038,54383000,11621600,57274600,41260600,173895000,6465800
2012-01-06,28.112152,157.199997,61.665424,51.117214,99.475777,80.689659,28.443141,158.630005,62.151885,51.232161,...,61.855059,50.949209,99.857445,80.023643,50333100,9790500,45499800,46325200,148050000,7348500
2012-01-09,28.406364,156.500000,61.978748,50.949203,99.717232,80.546921,28.465209,157.589996,62.085939,51.267526,...,61.954014,51.249841,99.701650,80.512944,45814800,8771900,52042400,39195500,99530200,5582400


In [7]:
prices_flat.columns[:15]

Index(['Close_EEM', 'Close_GLD', 'Close_IWM', 'Close_QQQ', 'Close_SPY',
       'Close_TLT', 'High_EEM', 'High_GLD', 'High_IWM', 'High_QQQ', 'High_SPY',
       'High_TLT', 'Low_EEM', 'Low_GLD', 'Low_IWM'],
      dtype='object')

In [8]:
print("Prices index checks:")
print(check_index_properties(prices))

print("\nMacro index checks:")
print(check_index_properties(macro))

Prices index checks:
{'index_type': 'DatetimeIndex', 'is_monotonic_increasing': True, 'has_duplicates': False, 'n_rows': 3519, 'start': Timestamp('2012-01-03 00:00:00'), 'end': Timestamp('2025-12-30 00:00:00')}

Macro index checks:
{'index_type': 'DatetimeIndex', 'is_monotonic_increasing': True, 'has_duplicates': False, 'n_rows': 950, 'start': Timestamp('1947-01-01 00:00:00'), 'end': Timestamp('2026-02-01 00:00:00')}


In [9]:
print("Duplicate dates in prices:", len(duplicate_dates(prices)))
print("Duplicate dates in macro:", len(duplicate_dates(macro)))

Duplicate dates in prices: 0
Duplicate dates in macro: 0


In [10]:
price_summary = summarize_dataframe(prices_flat, name="prices")
macro_summary = summarize_dataframe(macro, name="macro")

price_summary.sort_values("missing_count", ascending=False).head(15)

,name,dtype,missing_count,missing_pct
Close_EEM,prices,float64,0,0.0
Close_GLD,prices,float64,0,0.0
Volume_SPY,prices,int64,0,0.0
Volume_QQQ,prices,int64,0,0.0
Volume_IWM,prices,int64,0,0.0
Volume_GLD,prices,int64,0,0.0
Volume_EEM,prices,int64,0,0.0
Open_TLT,prices,float64,0,0.0
Open_SPY,prices,float64,0,0.0
Open_QQQ,prices,float64,0,0.0


In [11]:
macro_summary.sort_values("missing_count", ascending=False)

,name,dtype,missing_count,missing_pct
2y_treasury,macro,float64,353,37.157895
fed_funds,macro,float64,90,9.473684
10y_treasury,macro,float64,75,7.894737
unemployment,macro,float64,13,1.368421
cpi,macro,float64,1,0.105263


In [12]:
missing_by_row(prices_flat).head(10)

Date
2012-01-03    0
2021-05-06    0
2021-04-21    0
2021-04-22    0
2021-04-23    0
2021-04-26    0
2021-04-27    0
2021-04-28    0
2021-04-29    0
2021-04-30    0
dtype: int64

In [13]:
missing_by_row(macro).head(10)

date
1947-01-01    4
1947-08-01    4
1947-02-01    4
1947-12-01    4
1947-11-01    4
1947-09-01    4
1947-10-01    4
1947-07-01    4
1947-06-01    4
1947-05-01    4
dtype: int64

In [14]:
print("Large date gaps in prices:")
print(date_gaps(prices))

print("\nLarge date gaps in macro:")
print(date_gaps(macro).head(20))

Large date gaps in prices:
Series([], Name: Date, dtype: timedelta64[ns])

Large date gaps in macro:
date
1947-02-01   31 days
1947-03-01   28 days
1947-04-01   31 days
1947-05-01   30 days
1947-06-01   31 days
1947-07-01   30 days
1947-08-01   31 days
1947-09-01   31 days
1947-10-01   30 days
1947-11-01   31 days
1947-12-01   30 days
1948-01-01   31 days
1948-02-01   31 days
1948-03-01   29 days
1948-04-01   31 days
1948-05-01   30 days
1948-06-01   31 days
1948-07-01   30 days
1948-08-01   31 days
1948-09-01   31 days
Name: date, dtype: timedelta64[ns]


In [15]:
macro.head(10)

,fed_funds,cpi,unemployment,10y_treasury,2y_treasury
date,,,,,
1947-01-01,NaN,21.48,NaN,NaN,NaN
1947-02-01,NaN,21.62,NaN,NaN,NaN
1947-03-01,NaN,22.00,NaN,NaN,NaN
1947-04-01,NaN,22.00,NaN,NaN,NaN
1947-05-01,NaN,21.95,NaN,NaN,NaN
1947-06-01,NaN,22.08,NaN,NaN,NaN
1947-07-01,NaN,22.23,NaN,NaN,NaN
1947-08-01,NaN,22.40,NaN,NaN,NaN
1947-09-01,NaN,22.84,NaN,NaN,NaN


In [16]:
macro.tail(10)

,fed_funds,cpi,unemployment,10y_treasury,2y_treasury
date,,,,,
2025-05-01,4.33,320.620,4.3,4.42,3.92
2025-06-01,4.33,321.435,4.1,4.38,3.89
2025-07-01,4.33,322.169,4.3,4.39,3.88
2025-08-01,4.33,323.291,4.3,4.26,3.70
2025-09-01,4.22,324.245,4.4,4.12,3.57
2025-10-01,4.09,NaN,NaN,4.06,3.52
2025-11-01,3.88,325.063,4.5,4.09,3.55
2025-12-01,3.72,326.031,4.4,4.14,3.50
2026-01-01,3.64,326.588,4.3,4.21,3.54


In [17]:
check_index_properties(prices)
check_index_properties(macro)

{'index_type': 'DatetimeIndex',
 'is_monotonic_increasing': True,
 'has_duplicates': False,
 'n_rows': 950,
 'start': Timestamp('1947-01-01 00:00:00'),
 'end': Timestamp('2026-02-01 00:00:00')}

In [19]:
print(prices.shape)
print(macro.shape)

(3519, 30)
(950, 5)


In [20]:
prices_flat.columns[:15]

Index(['Close_EEM', 'Close_GLD', 'Close_IWM', 'Close_QQQ', 'Close_SPY',
       'Close_TLT', 'High_EEM', 'High_GLD', 'High_IWM', 'High_QQQ', 'High_SPY',
       'High_TLT', 'Low_EEM', 'Low_GLD', 'Low_IWM'],
      dtype='object')

In [21]:
macro.head()

,fed_funds,cpi,unemployment,10y_treasury,2y_treasury
date,,,,,
1947-01-01,NaN,21.48,NaN,NaN,NaN
1947-02-01,NaN,21.62,NaN,NaN,NaN
1947-03-01,NaN,22.00,NaN,NaN,NaN
1947-04-01,NaN,22.00,NaN,NaN,NaN
1947-05-01,NaN,21.95,NaN,NaN,NaN
